# Structured C3/C2/C1 biomedical answer judge

This notebook evaluates predictions using the question, supplied PubMed snippets, the prediction, and accepted BioASQ aliases.

- **C3** is assigned deterministically when the prediction matches an accepted alias under the fixed exact normalizer.
- For non-exact predictions, an independent LLM judge chooses **C2**, **C1**, or **UNCERTAIN**.
- The judge also reports semantic correctness, relatedness, evidence support, error type, and cited snippet IDs.

The output is a model-judged diagnostic. It should be calibrated against a small reviewed sample before being presented as validated semantic accuracy.

In [ ]:
from pathlib import Path
import os, json, re, hashlib, time
from collections import Counter

PROJECT_ROOT=Path('/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining')
EVAL_DIR=PROJECT_ROOT/'Artifacts/Factoid_SFT/evaluations/factoid_dev_qwen25_05b_full_dev_20260908_120106/evidence_per_alias_sft_dropout_strict_extractive'
INPUTS_FILE=EVAL_DIR/'evaluation_inputs.jsonl'
PREDICTIONS_FILE=EVAL_DIR/'per_question_metrics.json'
ALL_BIOASQ=PROJECT_ROOT/'data/training13b.json'
API_KEY_FILE=PROJECT_ROOT/'open_ai_api.txt'
JUDGE_ENDPOINT=os.environ.get('C3C1_JUDGE_ENDPOINT','https://api.openai.com/v1/chat/completions')
JUDGE_MODEL=os.environ.get('C3C1_JUDGE_MODEL','gpt-4.1-mini-2025-04-14')
OUTPUT_DIR=PROJECT_ROOT/'Artifacts/Factoid_SFT/class_judgments/c3_c1_v1'
SMOKE_TEST=False
SMOKE_QUESTIONS=1
JUDGE_EXACT_CASES=False
MAX_RETRIES=2
TIMEOUT_SECONDS=120
for p in [INPUTS_FILE,PREDICTIONS_FILE,ALL_BIOASQ,API_KEY_FILE]: assert p.exists(), p
print('Evaluation:',EVAL_DIR); print('Output:',OUTPUT_DIR); print('Judge:',JUDGE_MODEL)

In [ ]:
def load_jsonl(path):
    with Path(path).open(encoding='utf-8') as f: return [json.loads(x) for x in f if x.strip()]
def strip_tags(x): return re.sub(r'\s+',' ',re.sub(r'\[BE\]|\[EE\]',' ',str(x or ''),flags=re.I)).strip()
def exact_norm(x): return re.sub(r'[^a-z0-9]+','',strip_tags(x).replace('™','tm').replace('®','r').replace('µ','u').lower())
def digest(x): return hashlib.sha256(x.encode()).hexdigest()
def read_key(path):
    key=Path(path).read_text(encoding='utf-8').strip()
    if not key: raise ValueError('API key file is empty')
    return key
def extract_snippets(resources):
    snippets=[]
    for resource_index,res in enumerate(resources or [],1):
        res=str(res); m=re.search(r'PubMed ID:\s*([^\n]+)',res); pmid=m.group(1) if m else 'unknown'
        blocks=re.findall(r'\[BS\](.*?)\[ES\]',res,flags=re.S) or [res]
        for block_index,text in enumerate(blocks,1):
            text=re.sub(r'\s+',' ',text).strip()
            if text: snippets.append({'snippet_id':f'{resource_index}.{block_index}','pubmed_id':pmid,'text':text})
    return snippets
bio=json.loads(ALL_BIOASQ.read_text(encoding='utf-8'))['questions']
aliases_by_qid={str(q['id']):[str(a).strip() for a in (q.get('exact_answer') or []) if str(a).strip()] for q in bio if q.get('type')=='factoid'}
inputs={str(r['question_id']):r for r in load_jsonl(INPUTS_FILE)}
pred_rows=json.loads(PREDICTIONS_FILE.read_text(encoding='utf-8')); pred_rows=list(pred_rows.values()) if isinstance(pred_rows,dict) else pred_rows
records=[]
for row in pred_rows:
    qid=str(row['question_id']); inp=inputs.get(qid)
    if not inp: continue
    refs=aliases_by_qid.get(qid,[]) or [strip_tags(row.get('gold_output',''))]
    prediction=strip_tags(row.get('prediction',''))
    exact=any(exact_norm(prediction)==exact_norm(a) for a in refs if a)
    records.append({'question_id':qid,'question':inp.get('question',''),'gold_aliases':refs,'prediction':prediction,'snippets':extract_snippets(inp.get('resources',[])),'exact_match':exact,'source_model':row.get('model_name')})
print('Records:',len(records),'deterministic C3:',sum(r['exact_match'] for r in records))

## Judge rubric

The judge preserves the requested relation, entity, quantity, population, and essential qualifiers. Related concepts are not equivalent. Formatting alone is not a semantic error.

`evidence_support` is separate from the answer class: a semantically correct answer can still be unsupported by the supplied snippets.

In [ ]:
JUDGE_SYSTEM = r'''You are a biomedical factoid answer evaluator. Return JSON only.

Class definitions:
- C3: exact-correct. The candidate matches an accepted gold alias under the benchmark exact matcher. The caller normally assigns C3 deterministically.
- C2: equivalent-correct. The candidate is not an exact alias, but answers the question with the same biomedical entity, relation, value, population and essential qualifiers. Valid synonyms, abbreviations, nomenclature variants and numerical variants belong here.
- C1: incorrect or related-but-not-equivalent. The candidate does not answer the question equivalently to the accepted gold aliases. This includes unrelated, contradicted, hallucinated, wrong entity/value, non-answer, and also related but incorrect cases such as wrong relation, subtype, population, quantity, scope, broader/narrower concept, or wrong member.
- UNCERTAIN: the supplied information is insufficient for a confident C2/C1 decision.

Evaluate the candidate against the question and gold aliases. Use the supplied snippets to determine whether the candidate is supported. Do not penalize harmless wording or formatting. Do not infer a synonym merely because two terms are related. Cite only supplied snippet IDs. Keep the basis short; do not output hidden chain-of-thought.'''

def make_prompt(r):
    alias_text=' | '.join(r['gold_aliases'])
    snippet_text='\n'.join(f"Snippet {s['snippet_id']} (PubMed {s['pubmed_id']}): {s['text']}" for s in r['snippets'])
    return f'''Question: {r['question']}

Accepted gold aliases: {alias_text}

Candidate prediction: {r['prediction']}

Supplied snippets:
{snippet_text}

Return exactly this JSON schema:
{{
  "class": "C2",
  "semantic_correct": true,
  "related": true,
  "evidence_support": "supported",
  "evidence_ids": ["1.1"],
  "error_type": "none",
  "confidence": "high",
  "basis": "short explanation"
}}
Allowed class values: C2, C1, UNCERTAIN. Allowed evidence_support values: supported, unsupported, contradicted, insufficient. Allowed confidence values: high, medium, low.'''

def validate_judgment(j,r):
    if not isinstance(j,dict): raise ValueError('judge result is not an object')
    if j.get('class') not in {'C2','C1','C0','UNCERTAIN'}: raise ValueError('invalid class')
    if j.get('evidence_support') not in {'supported','unsupported','contradicted','insufficient'}: raise ValueError('invalid evidence_support')
    if j.get('confidence') not in {'high','medium','low'}: raise ValueError('invalid confidence')
    ids={s['snippet_id'] for s in r['snippets']}
    if not isinstance(j.get('evidence_ids'),list) or any(x not in ids for x in j['evidence_ids']): raise ValueError('invalid evidence_ids')
    if not isinstance(j.get('basis'),str) or not j['basis'].strip(): raise ValueError('missing basis')
    if j['class']=='C2' and j.get('semantic_correct') is not True: raise ValueError('C2 must be semantically correct')
    if j['class'] in {'C1','C0'} and j.get('semantic_correct') is not False: raise ValueError('C1/C0 must be semantically incorrect')
    return j

In [ ]:
def call_judge(prompt,api_key):
    import requests
    payload={'model':JUDGE_MODEL,'temperature':0,'max_tokens':500,'response_format':{'type':'json_object'},'messages':[{'role':'system','content':JUDGE_SYSTEM},{'role':'user','content':prompt}]}
    last=None
    for attempt in range(MAX_RETRIES+1):
        try:
            resp=requests.post(JUDGE_ENDPOINT,headers={'Authorization':f'Bearer {api_key}','Content-Type':'application/json'},json=payload,timeout=TIMEOUT_SECONDS)
            resp.raise_for_status(); return json.loads(resp.json()['choices'][0]['message']['content'])
        except Exception as exc:
            last=exc
            if attempt<MAX_RETRIES: time.sleep(2**attempt)
    raise RuntimeError(f'judge failed after retries: {last}')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True); cache=OUTPUT_DIR/'cache'; cache.mkdir(exist_ok=True)
api_key=read_key(API_KEY_FILE); work=records[:SMOKE_QUESTIONS] if SMOKE_TEST else records; judgments=[]
for i,r in enumerate(work,1):
    base={'question_id':r['question_id'],'prediction':r['prediction'],'exact_match':r['exact_match'],'source_model':r['source_model'],'input_sha256':digest(json.dumps(r,sort_keys=True,ensure_ascii=False))}
    if r['exact_match'] and not JUDGE_EXACT_CASES:
        out={**base,'class':'C3','semantic_correct':True,'related':True,'evidence_support':'not_judged','evidence_ids':[],'error_type':'none','confidence':'deterministic','basis':'Exact accepted-alias match.'}
    else:
        key=digest(json.dumps({'record':r,'system':JUDGE_SYSTEM,'model':JUDGE_MODEL},sort_keys=True,ensure_ascii=False)); cp=cache/f'{key}.json'
        try:
            if cp.exists(): out=json.loads(cp.read_text())
            else:
                j=call_judge(make_prompt(r),api_key); validate_judgment(j,r); out={**base,**j}; cp.write_text(json.dumps(out,ensure_ascii=False,indent=2))
        except Exception as exc:
            out={**base,'class':'UNCERTAIN','semantic_correct':None,'related':None,'evidence_support':'insufficient','evidence_ids':[],'error_type':'judge_error','confidence':'low','basis':str(exc)}
        if r['exact_match']: out={**out,'class':'C3','semantic_correct':True,'related':True,'error_type':'none','exact_override':True}
    judgments.append(out); print(i,'/',len(work),r['question_id'],out.get('class'),out.get('evidence_support'))
(OUTPUT_DIR/'judgments.jsonl').write_text(''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in judgments),encoding='utf-8')
counts=Counter(x['class'] for x in judgments); evidence=Counter(x.get('evidence_support') for x in judgments)
summary={'judged_records':len(judgments),'class_counts':dict(counts),'evidence_support_counts':dict(evidence),'smoke_test':SMOKE_TEST,'judge_model':JUDGE_MODEL,'rubric_sha256':digest(JUDGE_SYSTEM),'note':'C3 deterministic; C2/C1/C0/UNCERTAIN model-judged; evidence support is separate.'}
(OUTPUT_DIR/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8'); print(json.dumps(summary,indent=2))

The output files are `judgments.jsonl` and `summary.json`. For a full run, set `SMOKE_TEST=False`. Do not use `UNCERTAIN` as C0, and do not train on model-judged labels until a small stratified audit confirms C2/C1 behavior.